# ♻️ ClassifAI: MobileNetV2 Retraining Pipeline on TrashBusters Combined Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NithinGJ2005/CLassifAI/blob/main/notebooks/ClassifAI_MobileNetV2_Colab.ipynb)

This notebook provides a complete, reproducible retraining and evaluation workflow for **ClassifAI**.
- **Dataset:** `TrashBusters/combined` (Harmonized 6-class waste classification dataset from Hugging Face: 29,849 images across train, validation, and test splits).
- **Model Family:** MobileNetV2 with ImageNet-pretrained weights.
- **Workflow Highlights:**
  1. Dataset inspection, integrity checks, and split class distribution validation.
  2. Memory-safe streaming batch pipeline with prefetching and training data augmentation.
  3. **Phase 1:** Classifier head training with frozen MobileNetV2 backbone (Adam lr=0.001).
  4. **Phase 2:** Fine-tuning the top 30 layers with frozen BatchNormalization layers (Adam lr=1e-5).
  5. Model selection based on validation Macro-F1 score and loss.
  6. Comprehensive evaluation on held-out test split (macro/per-class precision, recall, F1, balanced accuracy, confusion matrix, latency benchmark).
  7. Export of production-ready `garbage_classifier.keras`, metadata, training curves, confusion matrix, and zipped package for deployment.

> **Colab GPU Setup Reminder:**  
> Before running, enable a GPU accelerator:  
> Go to **Runtime** > **Change runtime type** > **Hardware accelerator** > Select **T4 GPU** > Click **Save**.

In [ ]:
# ==============================================================================
# Step 1.1: Verify Hardware & Environment
# ==============================================================================
import os
import sys
import tensorflow as tf

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print("\n" + "="*70)
    print("⚠️  WARNING: NO GPU DETECTED!")
    print("Training on CPU will be extremely slow.")
    print("Please go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")
    print("="*70 + "\n")
else:
    print(f"\n✅ GPU is active: {gpus[0].name}")
    try:
        !nvidia-smi
    except Exception as e:
        print(f"Could not run nvidia-smi: {e}")


In [ ]:
# ==============================================================================
# Step 1.2: Install Required Dependencies
# ==============================================================================
!pip install -q datasets scikit-learn matplotlib pandas pillow seaborn


In [ ]:
# ==============================================================================
# Step 1.3: Imports, Reproducibility Seeds & Global Configuration
# ==============================================================================
import json
import time
import shutil
import random
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    accuracy_score
)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Global Configuration Constants
DATASET_NAME = "TrashBusters/combined"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 6
CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

# Artifact output directory
OUTPUT_DIR = "colab_artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration set successfully:")
print(f" - Dataset: {DATASET_NAME}")
print(f" - Input Shape: {IMAGE_SIZE + (3,)}")
print(f" - Batch Size: {BATCH_SIZE}")
print(f" - Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f" - Artifacts Directory: '{OUTPUT_DIR}'")


## Step 2: Dataset Loading, Exploration & Sanity Checks

We load `TrashBusters/combined` from Hugging Face Datasets.
We verify:
1. All 3 splits (`train`, `validation`, `test`) with sample counts.
2. Label mappings to ensure integer IDs 0 to 5 correspond precisely to `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`.
3. Class distribution across splits to check for class imbalances.
4. Representative image samples from each class.
5. Sanity checks (corrupt images, non-RGB formats) and potential cross-split duplicate checks.

In [ ]:
# ==============================================================================
# Step 2.1: Load Dataset from Hugging Face
# ==============================================================================
from datasets import load_dataset

print(f"Downloading/loading '{DATASET_NAME}' from Hugging Face...")
raw_dataset = load_dataset(DATASET_NAME)
print("\nDataset summary:")
print(raw_dataset)

# Verify expected splits
for split_name in ["train", "validation", "test"]:
    if split_name not in raw_dataset:
        raise ValueError(f"Required split '{split_name}' not found in dataset!")
    print(f" - Split '{split_name}': {len(raw_dataset[split_name]):,} images")


In [ ]:
# ==============================================================================
# Step 2.2: Verify Label Mapping & Taxonomy Consistency
# ==============================================================================
# Inspect sample items to ensure label integer matches CLASS_NAMES
sample_size = min(1000, len(raw_dataset['train']))
sample_data = raw_dataset['train'].select(range(sample_size))
unique_pairs = sorted(list(set((item['label'], item['label_name'].lower().strip()) for item in sample_data)))

print("Discovered (label_id, label_name) pairs:")
label_map = {}
for lid, lname in unique_pairs:
    print(f"  ID {lid} -> '{lname}'")
    label_map[lid] = lname

# Verify mapping against expected CLASS_NAMES order
for idx, name in enumerate(CLASS_NAMES):
    mapped_name = label_map.get(idx, None)
    if mapped_name != name:
        print(f"⚠️ Warning: ID {idx} is mapped to '{mapped_name}' but CLASS_NAMES[{idx}] is '{name}'")
    else:
        print(f"✅ ID {idx} correctly maps to '{name}'")

# Save class_names.json artifact
with open(os.path.join(OUTPUT_DIR, "class_names.json"), "w") as f:
    json.dump(CLASS_NAMES, f, indent=4)
print(f"\nSaved class_names.json to '{os.path.join(OUTPUT_DIR, 'class_names.json')}'")


In [ ]:
# ==============================================================================
# Step 2.3: Class Distribution Analysis
# ==============================================================================
dist_records = []
for split_name in ["train", "validation", "test"]:
    labels = raw_dataset[split_name]["label"]
    counts = pd.Series(labels).value_counts().sort_index()
    total = len(labels)
    for cls_idx, count in counts.items():
        cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f"Class_{cls_idx}"
        dist_records.append({
            "split": split_name,
            "class_id": cls_idx,
            "class_name": cls_name,
            "count": count,
            "percentage": (count / total) * 100
        })

dist_df = pd.DataFrame(dist_records)
pivot_counts = dist_df.pivot(index="class_name", columns="split", values="count").fillna(0).astype(int)
print("Class Distribution Summary (Counts):")
print(pivot_counts)

# Plot class distributions
plt.figure(figsize=(10, 5))
sns.barplot(data=dist_df, x="class_name", y="count", hue="split", palette="Set2")
plt.title("Class Distribution Across Dataset Splits", fontsize=13, fontweight="bold")
plt.xlabel("Waste Category", fontsize=11)
plt.ylabel("Number of Images", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=200)
plt.show()


In [ ]:
# ==============================================================================
# Step 2.4: Visualize Sample Images for Each Class
# ==============================================================================
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

found_classes = {}
for item in raw_dataset["train"]:
    lbl = item["label"]
    if lbl not in found_classes and lbl < NUM_CLASSES:
        found_classes[lbl] = item
    if len(found_classes) == NUM_CLASSES:
        break

for cls_idx in range(NUM_CLASSES):
    ax = axes[cls_idx]
    if cls_idx in found_classes:
        item = found_classes[cls_idx]
        img = item["image"]
        src = item.get("source_dataset", "Unknown")
        ctx = item.get("image_context", "")
        ax.imshow(img)
        title_str = f"[{cls_idx}] {CLASS_NAMES[cls_idx].upper()}\nSource: {src}"
        if ctx:
            title_str += f" ({ctx})"
        ax.set_title(title_str, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sample Training Images from TrashBusters/combined", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Step 2.5: Image Sanity Checks & Cross-Split Leakage / Duplicate Check
# ==============================================================================
print("Running sanity checks on images and labels...")

def check_split_integrity(split_ds, split_name, sample_size=1000):
    sample = split_ds.select(range(min(sample_size, len(split_ds))))
    corrupt_count = 0
    non_rgb_modes = set()
    for item in sample:
        img = item["image"]
        if img is None:
            corrupt_count += 1
        elif img.mode != "RGB":
            non_rgb_modes.add(img.mode)
    print(f"[{split_name}] Sampled {len(sample)} images:")
    print(f"  - Corrupt/None images: {corrupt_count}")
    print(f"  - Non-RGB color modes: {non_rgb_modes if non_rgb_modes else 'None (all RGB)'}")

for s in ["train", "validation", "test"]:
    check_split_integrity(raw_dataset[s], s)

# Fast duplicate check across splits using perceptual/MD5 hash
print("\nChecking for potential duplicate images across splits (sample test of 1,000 items)...")
def compute_img_hash(img):
    small = img.resize((32, 32)).convert("L")
    return hashlib.md5(small.tobytes()).hexdigest()

train_sample = raw_dataset["train"].select(range(min(3000, len(raw_dataset["train"]))))
val_sample = raw_dataset["validation"].select(range(min(1000, len(raw_dataset["validation"]))))
test_sample = raw_dataset["test"].select(range(min(1000, len(raw_dataset["test"]))))

train_hashes = {compute_img_hash(item["image"]): i for i, item in enumerate(train_sample)}
val_duplicates = sum(1 for item in val_sample if compute_img_hash(item["image"]) in train_hashes)
test_duplicates = sum(1 for item in test_sample if compute_img_hash(item["image"]) in train_hashes)

print(f"Duplicate matches in 1,000 sampled items:")
print(f" - Validation vs Train: {val_duplicates} / 1000")
print(f" - Test vs Train: {test_duplicates} / 1000")
if val_duplicates > 0 or test_duplicates > 0:
    print("ℹ️ Note: Small web-scraping overlaps are handled while keeping test split held-out.")
else:
    print("✅ No exact image hash collisions detected between sampled splits.")


## Step 3: Data Pipeline & Training Data Augmentation

- Target image dimensions: **224 × 224 pixels** with 3 RGB channels.
- Preprocessing: `tensorflow.keras.applications.mobilenet_v2.preprocess_input` (scales pixel values to $[-1, 1]$).
- **Moderate Data Augmentation (Training Only):**
  - Horizontal flip
  - Small rotation ($\pm 8\%$)
  - Moderate zoom ($\pm 10\%$)
  - Minor translation ($\pm 5\%$)
  - Mild contrast adjustment ($\pm 10\%$)
- Memory Safety: Batched generator stream reading from Arrow chunks with `AUTOTUNE` prefetching. Total RAM consumption remains under ~150 MB.

In [ ]:
# ==============================================================================
# Step 3.1: Define & Visualize Data Augmentation
# ==============================================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.08, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
    layers.RandomTranslation(0.05, 0.05, seed=SEED),
    layers.RandomContrast(0.1, seed=SEED),
], name="data_augmentation")

# Inspect augmentation visually on one sample image
sample_img = raw_dataset["train"][0]["image"].convert("RGB").resize(IMAGE_SIZE)
sample_tensor = tf.expand_dims(np.array(sample_img, dtype=np.float32) / 255.0, 0)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(sample_img)
axes[0].set_title("Original", fontsize=10, fontweight="bold")
axes[0].axis("off")

for i in range(1, 6):
    aug_img = data_augmentation(sample_tensor, training=True)[0].numpy()
    aug_img = np.clip(aug_img, 0.0, 1.0)
    axes[i].imshow(aug_img)
    axes[i].set_title(f"Aug Variant {i}", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Moderate Data Augmentation Verification (Applied to Training Set Only)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Step 3.2: Construct High-Efficiency tf.data.Dataset Pipeline
# ==============================================================================
def create_tf_dataset(hf_split, batch_size=BATCH_SIZE, is_training=False):
    num_samples = len(hf_split)
    
    def batch_generator():
        indices = np.arange(num_samples)
        if is_training:
            np.random.shuffle(indices)
            
        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_slice = indices[start_idx:end_idx]
            
            batch_data = hf_split[batch_slice]
            
            img_batch = []
            for img in batch_data["image"]:
                if img.mode != "RGB":
                    img = img.convert("RGB")
                if img.size != IMAGE_SIZE:
                    img = img.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
                img_batch.append(np.array(img, dtype=np.float32))
            
            # Apply MobileNetV2 preprocess_input to scale to [-1, 1]
            img_arr = preprocess_input(np.stack(img_batch, axis=0))
            labels_arr = np.array(batch_data["label"], dtype=np.int32)
            
            yield img_arr, labels_arr

    output_signature = (
        tf.TensorSpec(shape=(None, IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)
    )
    
    ds = tf.data.Dataset.from_generator(batch_generator, output_signature=output_signature)
    
    if is_training:
        # Augmentation on training batches
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
        
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_tf_dataset(raw_dataset["train"], batch_size=BATCH_SIZE, is_training=True)
val_ds = create_tf_dataset(raw_dataset["validation"], batch_size=BATCH_SIZE, is_training=False)
test_ds = create_tf_dataset(raw_dataset["test"], batch_size=BATCH_SIZE, is_training=False)

train_steps = int(np.ceil(len(raw_dataset["train"]) / BATCH_SIZE))
val_steps = int(np.ceil(len(raw_dataset["validation"]) / BATCH_SIZE))
test_steps = int(np.ceil(len(raw_dataset["test"]) / BATCH_SIZE))

print("Data pipelines initialized:")
print(f" - Train: {len(raw_dataset['train']):,} samples -> {train_steps} batches/epoch")
print(f" - Validation: {len(raw_dataset['validation']):,} samples -> {val_steps} batches/epoch")
print(f" - Test: {len(raw_dataset['test']):,} samples -> {test_steps} batches")


## Step 4: Phase 1 — Train the Classifier Head (Frozen MobileNetV2 Backbone)

- **Base Model:** `MobileNetV2` with ImageNet-pretrained weights, frozen (`base_model.trainable = False`).
- **Preserved Head:**
  - `GlobalAveragePooling2D()`
  - `Dense(256, activation='relu')`
  - `Dropout(0.5)`
  - `Dense(6, activation='softmax')`
- **Optimizer:** `Adam(learning_rate=0.001)`
- **Loss:** `SparseCategoricalCrossentropy()`
- **Callbacks:** EarlyStopping (patience=4), ReduceLROnPlateau, ModelCheckpoint.

In [ ]:
# ==============================================================================
# Step 4.1: Construct MobileNetV2 with Classifier Head
# ==============================================================================
tf.keras.backend.clear_session()

base_model = MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze backbone for Phase 1

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name="input_image")
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
x = layers.Dense(256, activation="relu", name="dense_256")(x)
x = layers.Dropout(0.5, name="dropout_0.5")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output_probabilities")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="ClassifAI_MobileNetV2")

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# ==============================================================================
# Step 4.2: Execute Phase 1 Training
# ==============================================================================
phase1_checkpoint_path = os.path.join(OUTPUT_DIR, "phase1_best_model.keras")

phase1_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=phase1_checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

EPOCHS_PHASE1 = 20
print(f"Starting Phase 1 Training (Max {EPOCHS_PHASE1} epochs)...\n")
start_time_p1 = time.time()

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=phase1_callbacks
)

time_p1 = time.time() - start_time_p1
print(f"\n✅ Phase 1 Training completed in {time_p1/60:.2f} minutes.")


## Step 5: Phase 2 — Fine-Tuning MobileNetV2 (Top 30 Layers)

- Starting from the best checkpoint obtained in Phase 1.
- Unfreezing the final 30 layers of MobileNetV2.
- **BatchNormalization layers are strictly frozen** (`layer.trainable = False`) to prevent unstable moving mean/variance updates.
- Recompiling with a low learning rate: `Adam(learning_rate=1e-5)`.
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint (`phase2_best_model.keras`).
- Model selection on the held-out **Validation Set**, comparing Phase 1 vs Phase 2 on Macro-F1 and validation loss.

In [ ]:
# ==============================================================================
# Step 5.1: Unfreeze Top 30 Layers (Batch Normalization Frozen)
# ==============================================================================
base_model.trainable = True

# Unfreeze only the top 30 layers
fine_tune_at = len(base_model.layers) - 30
print(f"Total base_model layers: {len(base_model.layers)}")
print(f"Freezing all base layers prior to layer {fine_tune_at}...")

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Strictly freeze BatchNormalization layers
bn_count = 0
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
        bn_count += 1

print(f"Frozen {bn_count} BatchNormalization layers to preserve stability.")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print(f"Trainable tensors: {len(model.trainable_weights)}, Non-trainable: {len(model.non_trainable_weights)}")


In [ ]:
# ==============================================================================
# Step 5.2: Execute Phase 2 Fine-Tuning
# ==============================================================================
phase2_checkpoint_path = os.path.join(OUTPUT_DIR, "phase2_best_model.keras")

phase2_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=phase2_checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

EPOCHS_PHASE2 = 15
print(f"Starting Phase 2 Fine-Tuning (Max {EPOCHS_PHASE2} epochs)...\n")
start_time_p2 = time.time()

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=phase2_callbacks
)

time_p2 = time.time() - start_time_p2
print(f"\n✅ Phase 2 Fine-Tuning completed in {time_p2/60:.2f} minutes.")


In [ ]:
# ==============================================================================
# Step 5.3: Model Selection on Validation Set (Phase 1 vs Phase 2)
# ==============================================================================
print("Evaluating candidate models on held-out Validation Set...")

# Load Phase 1 checkpoint
m_p1 = tf.keras.models.load_model(phase1_checkpoint_path)
val_loss_p1, val_acc_p1 = m_p1.evaluate(val_ds, steps=val_steps, verbose=0)

# Evaluate Phase 2 checkpoint
if os.path.exists(phase2_checkpoint_path):
    m_p2 = tf.keras.models.load_model(phase2_checkpoint_path)
else:
    m_p2 = model
val_loss_p2, val_acc_p2 = m_p2.evaluate(val_ds, steps=val_steps, verbose=0)

def compute_val_macro_f1(cand_model):
    preds_list = []
    targets_list = []
    for x_b, y_b in val_ds.take(val_steps):
        p = cand_model.predict(x_b, verbose=0)
        preds_list.extend(np.argmax(p, axis=1))
        targets_list.extend(y_b.numpy())
    return precision_recall_fscore_support(targets_list, preds_list, average="macro")[2]

val_f1_p1 = compute_val_macro_f1(m_p1)
val_f1_p2 = compute_val_macro_f1(m_p2)

comparison_df = pd.DataFrame([
    {"Candidate": "Phase 1 (Frozen Backbone)", "Val Loss": val_loss_p1, "Val Accuracy": val_acc_p1, "Val Macro-F1": val_f1_p1},
    {"Candidate": "Phase 2 (Fine-Tuned Top 30)", "Val Loss": val_loss_p2, "Val Accuracy": val_acc_p2, "Val Macro-F1": val_f1_p2}
])
print("\nValidation Performance Comparison:")
print(comparison_df.round(4).to_string(index=False))

# Select best candidate based on macro-F1
if val_f1_p2 > val_f1_p1 and val_loss_p2 <= val_loss_p1 * 1.05:
    print("\n🏆 Selected Candidate: Phase 2 Fine-Tuned Model")
    selected_model = m_p2
    selected_candidate_name = "Phase 2 Fine-Tuned"
else:
    print("\n🏆 Selected Candidate: Phase 1 Frozen Backbone Model")
    selected_model = m_p1
    selected_candidate_name = "Phase 1 Frozen Backbone"

# Save the winning model as garbage_classifier.keras
final_model_path = os.path.join(OUTPUT_DIR, "garbage_classifier.keras")
selected_model.save(final_model_path)
print(f"\nProduction model saved to: '{final_model_path}'")


## Step 6: Comprehensive Model Evaluation (Held-Out Test Split)

We evaluate the selected model against the completely held-out **Test split** (`4,478 images`).
Evaluation includes:
- Overall Accuracy & Balanced Accuracy
- Test Loss
- Per-Class and Macro Precision, Recall, F1-scores
- Inference Latency Benchmark (ms/image)
- Confusion Matrix Heatmaps (Raw counts and Normalized percentages)
- Combined Training and Validation Curves
- Misclassified Image Gallery
- Sub-Dataset Generalization Analysis (`source_dataset`)
- Optional head-to-head comparison with legacy `garbage_classifier.h5`

In [ ]:
# ==============================================================================
# Step 6.1: Run Inference & Compute Comprehensive Test Metrics
# ==============================================================================
print("Evaluating selected model on held-out Test split...")

test_loss, test_acc = selected_model.evaluate(test_ds, steps=test_steps, verbose=1)

all_preds = []
all_probs = []
all_targets = []
inference_times = []

for x_batch, y_batch in test_ds.take(test_steps):
    t0 = time.time()
    probs = selected_model.predict(x_batch, verbose=0)
    latency = (time.time() - t0) / len(x_batch)
    inference_times.append(latency)
    
    preds = np.argmax(probs, axis=1)
    all_probs.extend(probs)
    all_preds.extend(preds)
    all_targets.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

# Compute metrics
avg_latency_ms = float(np.mean(inference_times) * 1000)
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro")
weighted_prec, weighted_rec, weighted_f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="weighted")
bal_acc = float(balanced_accuracy_score(all_targets, all_preds))

prec_per_class, rec_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(all_targets, all_preds, average=None)

metrics_dict = {
    "selected_model": selected_candidate_name,
    "test_loss": float(test_loss),
    "overall_accuracy": float(test_acc),
    "balanced_accuracy": bal_acc,
    "macro_precision": float(macro_prec),
    "macro_recall": float(macro_rec),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "average_latency_ms_per_image": avg_latency_ms,
    "per_class": {
        CLASS_NAMES[i]: {
            "precision": float(prec_per_class[i]),
            "recall": float(rec_per_class[i]),
            "f1_score": float(f1_per_class[i]),
            "support": int(support_per_class[i])
        } for i in range(NUM_CLASSES)
    }
}

metrics_json_path = os.path.join(OUTPUT_DIR, "test_metrics.json")
with open(metrics_json_path, "w") as f:
    json.dump(metrics_dict, f, indent=4)

print("\n" + "="*50)
print("FINAL TEST RESULTS (Held-Out Split):")
print("="*50)
print(f"Selected Model:     {selected_candidate_name}")
print(f"Overall Accuracy:   {test_acc*100:.2f}%")
print(f"Balanced Accuracy:  {bal_acc*100:.2f}%")
print(f"Macro F1-Score:     {macro_f1:.4f}")
print(f"Weighted F1-Score:  {weighted_f1:.4f}")
print(f"Test Loss:          {test_loss:.4f}")
print(f"Inference Latency:  {avg_latency_ms:.2f} ms/image")
print("="*50)

# Save Classification Report CSV
clf_report = classification_report(all_targets, all_preds, target_names=CLASS_NAMES, output_dict=True)
clf_report_df = pd.DataFrame(clf_report).transpose()
clf_report_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report.csv"))

print("\nClassification Report:")
print(clf_report_df.round(4).to_string())


In [ ]:
# ==============================================================================
# Step 6.2: Plot & Save Combined Training Curves
# ==============================================================================
h1 = history_phase1.history
h2 = history_phase2.history if 'history_phase2' in locals() else None

loss = h1["loss"] + (h2["loss"] if h2 else [])
val_loss = h1["val_loss"] + (h2["val_loss"] if h2 else [])
acc = h1["accuracy"] + (h2["accuracy"] if h2 else [])
val_acc = h1["val_accuracy"] + (h2["val_accuracy"] if h2 else [])

history_df = pd.DataFrame({
    "epoch": range(1, len(loss) + 1),
    "loss": loss,
    "val_loss": val_loss,
    "accuracy": acc,
    "val_accuracy": val_acc
})
history_df.to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)

plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.plot(loss, label="Training Loss", color="#2563eb", linewidth=2)
plt.plot(val_loss, label="Validation Loss", color="#dc2626", linewidth=2)
if h2:
    plt.axvline(x=len(h1["loss"])-1, color="gray", linestyle="--", label="Fine-Tuning Start")
plt.title("Loss Curves", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Loss", fontsize=10)
plt.legend(frameon=True)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(acc, label="Training Accuracy", color="#2563eb", linewidth=2)
plt.plot(val_acc, label="Validation Accuracy", color="#16a34a", linewidth=2)
if h2:
    plt.axvline(x=len(h1["loss"])-1, color="gray", linestyle="--", label="Fine-Tuning Start")
plt.title("Accuracy Curves", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Accuracy", fontsize=10)
plt.legend(frameon=True)
plt.grid(True, alpha=0.3)

plt.tight_layout()
training_curves_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(training_curves_path, dpi=200)
plt.show()
print(f"Saved curves to '{training_curves_path}'")


In [ ]:
# ==============================================================================
# Step 6.3: Plot & Save Confusion Matrix
# ==============================================================================
cm = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title("Test Confusion Matrix (Counts)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Predicted Label", fontsize=10)
ax1.set_ylabel("True Label", fontsize=10)

sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
ax2.set_title("Test Confusion Matrix (Normalized Recall)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Predicted Label", fontsize=10)
ax2.set_ylabel("True Label", fontsize=10)

plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=200)
plt.show()
print(f"Saved confusion matrix to '{cm_path}'")


In [ ]:
# ==============================================================================
# Step 6.4: Inspect Misclassified Test Images
# ==============================================================================
misclassified_indices = np.where(all_preds != all_targets)[0]
pct_misclassified = (len(misclassified_indices) / len(all_targets)) * 100
print(f"Total misclassified test images: {len(misclassified_indices):,} / {len(all_targets):,} ({pct_misclassified:.2f}%)")

num_to_show = min(6, len(misclassified_indices))
if num_to_show > 0:
    fig, axes = plt.subplots(1, num_to_show, figsize=(16, 3.2))
    if num_to_show == 1:
        axes = [axes]
    for i in range(num_to_show):
        idx = misclassified_indices[i]
        item = raw_dataset["test"][int(idx)]
        img = item["image"]
        true_name = CLASS_NAMES[all_targets[idx]]
        pred_name = CLASS_NAMES[all_preds[idx]]
        conf = all_probs[idx][all_preds[idx]] * 100
        
        axes[i].imshow(img)
        axes[i].set_title(f"True: {true_name}\nPred: {pred_name} ({conf:.1f}%)", color="#dc2626", fontsize=9, fontweight="bold")
        axes[i].axis("off")
    plt.suptitle("Sample Misclassified Test Items", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# Step 6.5: Performance Breakdown by Source Dataset
# ==============================================================================
if "source_dataset" in raw_dataset["test"].column_names:
    test_sources = raw_dataset["test"]["source_dataset"]
    eval_df = pd.DataFrame({
        "source": test_sources,
        "true": all_targets,
        "pred": all_preds,
        "correct": (all_targets == all_preds).astype(int)
    })
    
    source_summary = eval_df.groupby("source").agg(
        total_images=("correct", "count"),
        accuracy=("correct", "mean")
    ).reset_index()
    source_summary["accuracy_pct"] = (source_summary["accuracy"] * 100).round(2)
    
    print("Generalization Across Source Datasets:")
    print(source_summary[["source", "total_images", "accuracy_pct"]].to_string(index=False))


In [ ]:
# ==============================================================================
# Step 6.6: Optional Head-to-Head Comparison with Legacy Baseline Model
# ==============================================================================
# If you upload artifacts/models/garbage_classifier.h5 to Colab, this cell will evaluate it
baseline_path = "garbage_classifier.h5"

if os.path.exists(baseline_path):
    print(f"Found legacy baseline model at '{baseline_path}'. Running evaluation...")
    try:
        baseline_model = tf.keras.models.load_model(baseline_path)
        b_loss, b_acc = baseline_model.evaluate(test_ds, steps=test_steps, verbose=0)
        
        b_preds = []
        for x_b, _ in test_ds.take(test_steps):
            p = baseline_model.predict(x_b, verbose=0)
            b_preds.extend(np.argmax(p, axis=1))
        
        b_macro_f1 = precision_recall_fscore_support(all_targets, b_preds, average="macro")[2]
        
        comp_df = pd.DataFrame([
            {"Model": "Legacy Baseline (.h5)", "Test Accuracy": f"{b_acc*100:.2f}%", "Macro-F1": f"{b_macro_f1:.4f}", "Test Loss": f"{b_loss:.4f}"},
            {"Model": "New Model (.keras)", "Test Accuracy": f"{test_acc*100:.2f}%", "Macro-F1": f"{macro_f1:.4f}", "Test Loss": f"{test_loss:.4f}"}
        ])
        print("\nHead-to-Head Comparison:")
        print(comp_df.to_string(index=False))
    except Exception as e:
        print(f"Could not evaluate baseline model: {e}")
else:
    print("ℹ️ Legacy 'garbage_classifier.h5' not found in working directory.")
    print("To compare, upload 'garbage_classifier.h5' to Colab and re-run this cell.")


## Step 7: Artifact Export, Packaging & Download

All deliverables are stored in `colab_artifacts/`:
- `garbage_classifier.keras`: Trained production model
- `class_names.json`: Authoritative class order list
- `test_metrics.json`: Full evaluation metrics
- `classification_report.csv`: Per-class precision, recall, and support
- `confusion_matrix.png`: Normalized and count confusion matrix
- `training_curves.png`: Training loss and accuracy plots
- `training_history.csv`: Per-epoch logs
- `evaluation_summary.md`: Markdown summary report

Run the cells below to bundle everything into `classifai_trained_artifacts.zip` and download it directly to your computer.

In [ ]:
# ==============================================================================
# Step 7.1: Generate Summary Report & Package Zip Archive
# ==============================================================================
summary_md = f"""# ClassifAI Model Retraining Evaluation Summary

- **Date:** {time.strftime('%Y-%m-%d %H:%M:%S')}
- **Dataset:** `{DATASET_NAME}`
- **Selected Architecture:** `{selected_candidate_name}`
- **Test Split Count:** {len(all_targets):,} images

## Overall Test Metrics
- **Test Accuracy:** {test_acc*100:.2f}%
- **Balanced Accuracy:** {bal_acc*100:.2f}%
- **Macro Precision:** {macro_prec:.4f}
- **Macro Recall:** {macro_rec:.4f}
- **Macro F1-Score:** {macro_f1:.4f}
- **Weighted F1-Score:** {weighted_f1:.4f}
- **Test Loss:** {test_loss:.4f}
- **Average Latency:** {avg_latency_ms:.2f} ms/image

## Per-Class Results
"""
for c in CLASS_NAMES:
    pc = metrics_dict["per_class"][c]
    summary_md += f"- **{c.capitalize()}:** Precision={pc['precision']:.4f}, Recall={pc['recall']:.4f}, F1={pc['f1_score']:.4f}, Support={pc['support']}\n"

with open(os.path.join(OUTPUT_DIR, "evaluation_summary.md"), "w", encoding="utf-8") as f:
    f.write(summary_md)

# Create zip archive
zip_base_name = "classifai_trained_artifacts"
zip_path = shutil.make_archive(zip_base_name, "zip", OUTPUT_DIR)
print(f"✅ Successfully created archive: '{zip_path}'")

# Print file sizes
print("\nPackaged Artifacts:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    sz_mb = os.path.getsize(fp) / (1024 * 1024)
    print(f"  - {f:30s} ({sz_mb:.2f} MB)")


In [ ]:
# ==============================================================================
# Step 7.2: Download Artifacts to Local Machine
# ==============================================================================
from google.colab import files

print("Triggering browser download for 'classifai_trained_artifacts.zip'...")
try:
    files.download(f"{zip_base_name}.zip")
    print("Download initiated! Check your browser's download folder.")
except Exception as e:
    print(f"Could not trigger automated download: {e}")
    print(f"You can manually right-click '{zip_base_name}.zip' in the Colab file tree on the left and select Download.")


In [ ]:
# ==============================================================================
# Step 7.3: (Optional) Backup Directly to Google Drive
# ==============================================================================
# Uncomment and run if you want to store artifacts in Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# drive_target_dir = '/content/drive/MyDrive/ClassifAI_Artifacts'
# os.makedirs(drive_target_dir, exist_ok=True)
# shutil.copy(f'{zip_base_name}.zip', drive_target_dir)
# print(f'Successfully copied to {drive_target_dir}')
